## Weather-AQI MCP Assistant
**`Weather-AQI MCP Assistant`** is an interactive, asynchronous assistant that brings together real-time weather and AQI (Air Quality Index) data using powerful **MCP** (Model Context Protocol) servers. There are 3 MCP servers that are created using **`FastMCP`** which is a high-level, Pythonic framework inspired by FastAPI that simplifies MCP implementation.\
It seamlessly connects to dedicated weather and AQI tools on **Intel® Core™ Ultra Processors** then uses [**Qwen/Qwen2.5-3B-Instruct**](https://huggingface.co/Qwen/Qwen2.5-3B-Instruct) to analyze the data and generate clear, actionable health and safety recommendations. The Qwen2.5-3B-Instruct model is loaded using the [**PyTorch XPU backend**](https://docs.pytorch.org/docs/stable/notes/get_start_xpu.html) to leverage Intel hardware acceleration.This assistant helps users stay informed about environmental conditions and make better decisions for their well-being. Designed with async operations and SSE connections, it’s perfect for modern, event-driven pipelines

## Workflow
This diagram illustrates how the Weather-AQI MCP Assistant operates end-to-end within an AI PC environment, combining MCP Compliant Servers, a client, and external APIs. The servers are created using **`FastMCP`** which is a high-level, **`Pythonic framework`** inspired by FastAPI that simplifies MCP implementation.

1. **Air Quality Index(AQI) server:** Returns AQI data and air pollutant levels of the input location.
2. **Weather server:** Returns Weather report like Temparature and wind speed in the user specified location.
3. **LLM Inferencing server:** Based on the weather and AQI reports generated from the above servers, the model generates Health advice and safety guidelines and necessary precautions accordingly.
4. **Weather & AQI Advisory:** This notebook (acts as MCP Client) coordinates the workflow between these 3 servers "

## Import necessary packages
Import all the necessary packages and libraries including Gradio for the UI. Apply nest_asyncio to allow nested event loops

In [1]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import os
from fastmcp import Client
import gradio as gr

## Define the MCP Client class AQI_Weather_Advisor
This class contains definitions of async methods i.e. `get_weather()`, `get_aqi_report()` and `get_health_recommendations()` that calls corresponding FastMCP tools and return results asynchronously. The `extract_text` helper normalizes different output formats.

In [2]:
import asyncio
from fastmcp import Client

class AQI_Weather_Advisor:
    """
    AQI_Weather_Advisor is responsible for interacting with Weather, AQI, and LLM MCP servers.

    It provides methods to:
    - Fetch weather information for a given location.
    - Retrieve AQI (Air Quality Index) reports.
    - Generate health and safety recommendations based on weather and AQI data.
    """

    def __init__(self, weather_url: str, aqi_server_url: str, llm_server_url: str):
        """
        Initialize the AQI_Weather_Advisor with URLs for Weather, AQI, and LLM MCP servers.

        Args:
            weather_url (str): URL of the Weather MCP server.
            aqi_server_url (str): URL of the AQI MCP server.
            llm_server_url (str): URL of the LLM MCP server.
        """
        self.weather_url = weather_url
        self.aqi_server_url = aqi_server_url
        self.llm_server_url = llm_server_url

    async def get_weather(self, location: str) -> str:
        """
        Retrieve weather data for a given location from the Weather MCP server.

        Args:
            location (str): Name of the location to get weather information for.

        Returns:
            str: Raw weather data response or error message.
        """
        try:
            async with Client(f"{self.weather_url}/sse") as client:
                return await client.call_tool("get_weather", {"location": location})
        except Exception as e:
            return f" Failed to get weather data for '{location}': {str(e)}"

    async def get_aqi_report(self, location: str) -> str:
        """
        Retrieve AQI (Air Quality Index) report for a given location from the AQI MCP server.

        Args:
            location (str): Name of the location to get AQI report for.

        Returns:
            str: AQI report as plain text or error message.
        """
        try:
            async with Client(f"{self.aqi_server_url}/sse") as client:
                result = await client.call_tool("get_aqi", {"location": location})
                return self._extract_text(result)
        except Exception as e:
            return f" Failed to get AQI report for '{location}': {str(e)}"

    async def get_health_recommendations(self, weather_report: str, aqi_report: str) -> str:
        """
        Get health and safety recommendations by calling the LLM MCP server.

        Args:
            weather_report (str): Weather report text.
            aqi_report (str): AQI report text.

        Returns:
            str: Health and safety recommendations or error message.
        """
        try:
            async with Client(f"{self.llm_server_url}/sse") as client:
                result = await client.call_tool("safety_guidelines", {
                    "weather_report": weather_report,
                    "aqi_report": aqi_report
                })
                return self._extract_text(result)
        except Exception as e:
            return f" Failed to get safety recommendations: {str(e)}"

    def _extract_text(self, result) -> str:
        """
        Helper method to extract plain text from MCP results.

        Args:
            result (Any): The result returned from an MCP tool call.

        Returns:
            str: Extracted text or string representation.
        """
        try:
            if isinstance(result, list):
                return "\n".join(block.text for block in result if hasattr(block, "text"))
            elif hasattr(result, "text"):
                return result.text
            return str(result)
        except Exception as e:
            return f" Failed to parse result: {str(e)}"


## Create Gradio Interface
Build an interactive Gradio UI for the Weather-AQI MCP Assistant that:
- Accepts city input (required) with optional state/province and country for disambiguation
- Displays weather, AQI, and health recommendations in separate output boxes
- Provides a clean, user-friendly interface with helpful examples
- Includes pre-populated example to demonstrate usage

In [ ]:
# Initialize the advisor
agent = AQI_Weather_Advisor("http://127.0.0.1:8000", "http://127.0.0.1:8001", "http://127.0.0.1:8002")

async def get_weather_aqi_report(city: str, state: str = "", country: str = "", progress=gr.Progress()):
    """
    Fetch weather, AQI, and health recommendations for a given location.
    Runs Weather and AQI requests in parallel for faster responses.
    
    Args:
        city: Name of the city
        state: Optional state/province (e.g., "Oregon", "Maine")
        country: Optional country code (e.g., "US", "CA")
        progress: Gradio progress tracker
        
    Returns:
        Tuple of (weather_report, aqi_report, health_recommendations)
    """
    if not city or not city.strip():
        return "⚠️ Please enter a city", "", ""
    
    # Build location string with optional state/country
    location_parts = [city.strip()]
    if state and state.strip():
        location_parts.append(state.strip())
    if country and country.strip():
        location_parts.append(country.strip())
    
    location = ", ".join(location_parts)
    
    try:
        # Step 1: Fetch Weather and AQI in parallel (saves 1-2 seconds)
        progress(0.1, desc=f"🔍 Looking up {location}...")
        
        weather_task = agent.get_weather(location)
        aqi_task = agent.get_aqi_report(location)
        
        progress(0.3, desc="🌡️ Fetching weather data...")
        progress(0.4, desc="💨 Fetching air quality data...")
        
        # Run both requests concurrently
        weather_raw, aqi_report = await asyncio.gather(weather_task, aqi_task)
        
        weather_report = weather_raw[0].text if isinstance(weather_raw, list) else str(weather_raw)
        
        progress(0.6, desc="✅ Got weather & AQI data")
        
        # Step 2: Get health recommendations from LLM
        progress(0.7, desc="🤖 Generating health recommendations...")
        recommendations = await agent.get_health_recommendations(weather_report, aqi_report)
        
        progress(1.0, desc="✅ Complete!")
        
        return weather_report, aqi_report, recommendations
        
    except Exception as e:
        error_msg = f"❌ Error: {str(e)}"
        return error_msg, "", ""

def gradio_wrapper(city: str, state: str, country: str, progress=gr.Progress()):
    """Synchronous wrapper for Gradio to call async function"""
    return asyncio.run(get_weather_aqi_report(city, state, country, progress))

# Create Gradio interface
with gr.Blocks(title="Weather & AQI Assistant", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        """
        # 🌤️ Weather & AQI Health Assistant
        
        Get real-time weather, air quality data, and personalized health recommendations 
        powered by MCP servers and AI on Intel® Core™ Ultra Processors.
        
        ### 📝 How to Use
        - **🏙️ City** (Required): Enter the city name
        - **🗺️ State/Province** (Optional): For cities with common names (e.g., Portland in Oregon vs Maine)
        - **🌍 Country** (Optional): Use ISO 2-letter codes (US, CA, GB, FR, DE, JP, etc.)
        
        **Example:** Hillsboro, Oregon, US
        """
    )
    
    with gr.Row():
        with gr.Column():
            city_input = gr.Textbox(
                label="🏙️ City (Required)",
                placeholder="e.g., Hillsboro, Portland, Tokyo...",
                value="Hillsboro",
                lines=1
            )
            with gr.Row():
                state_input = gr.Textbox(
                    label="🗺️ State/Province (Optional)",
                    placeholder="e.g., Oregon, Maine, California...",
                    value="Oregon",
                    lines=1,
                    scale=2
                )
                country_input = gr.Textbox(
                    label="🌍 Country Code (Optional)",
                    placeholder="US, CA, GB, FR, DE, JP...",
                    value="US",
                    lines=1,
                    scale=1
                )
            submit_btn = gr.Button("🔍 Get Report", variant="primary", size="lg")
            
            gr.Markdown(
                """
                **💡 Tips:**
                - Leave state/country blank for major cities (e.g., Tokyo, Paris)
                - Specify state for common city names (Portland, Springfield, etc.)
                - Use 2-letter country codes (ISO 3166-1 alpha-2)
                """
            )
    
    gr.Markdown("---")
    
    with gr.Row():
        with gr.Column():
            weather_output = gr.Textbox(
                label="🌡️ Weather Report",
                lines=8,
                interactive=False
            )
        
        with gr.Column():
            aqi_output = gr.Textbox(
                label="💨 Air Quality Index (AQI)",
                lines=8,
                interactive=False
            )
    
    with gr.Row():
        health_output = gr.Textbox(
            label="🏥 Health & Safety Recommendations",
            lines=10,
            interactive=False
        )
    
    gr.Markdown(
        """
        ---
        ### ⚙️ About the MCP Servers
        This assistant uses three MCP (Model Context Protocol) servers running locally:
        - **Weather Server** (port 8000): Real-time weather data from Open-Meteo
        - **AQI Server** (port 8001): Air quality measurements from OpenWeatherMap
        - **LLM Server** (port 8002): AI-powered health recommendations using Intel® OpenVINO™
        
        **⚡ Performance:** Weather and AQI data are fetched in parallel for faster responses.
        """
    )
    
    # Connect the button click to the function
    submit_btn.click(
        fn=gradio_wrapper,
        inputs=[city_input, state_input, country_input],
        outputs=[weather_output, aqi_output, health_output],
        show_progress=True
    )
    
    # Also allow Enter key to submit from city input
    city_input.submit(
        fn=gradio_wrapper,
        inputs=[city_input, state_input, country_input],
        outputs=[weather_output, aqi_output, health_output],
        show_progress=True
    )

## Launch the Gradio App

Launch the interactive Gradio web interface. The app will open in a new browser tab at [http://127.0.0.1:7860](http://127.0.0.1:7860) or display inline in the notebook.

In [4]:
# Launch the Gradio app
# Opens in a new browser tab at http://127.0.0.1:7860
demo.launch(share=False, server_name="127.0.0.1")

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
